# 07 · Decorators — **Depth**

> **Pull this notebook when:** you want cross-cutting behavior — caching, timing, validation,
> registration — without editing the wrapped function. The component registry in Acts 1–2 is built
> from this.

Depth here is what `@` *desugars to* and how the layers stack. Built on the closures from notebook 02.
Predict first.

---

## 7.1 — Predict: `@deco` is just reassignment

```python
def deco(func):
    def wrapper(*a, **k):
        return func(*a, **k) * 2
    return wrapper

@deco
def f(x):
    return x + 1

# the @ line above is EXACTLY equivalent to: f = deco(f)
print(f(10))           # ?
```

In [ ]:
def deco(func):
    def wrapper(*a, **k):
        return func(*a, **k) * 2
    return wrapper

@deco
def f(x):
    return x + 1

print(f(10))

<details>
<summary>▶ Reveal</summary>

`22`.

`@deco` above `f` is pure sugar for `f = deco(f)` *after* the definition. So `f` is no longer the
original — it's `wrapper`, which calls the original (`10 + 1 = 11`) and doubles it (`22`). The whole
"magic" of decorators is this one substitution: define the function, pass it through the decorator,
rebind the name to the result. Everything else is just closures (notebook 02) — `wrapper` closes over
`func`.

</details>

## 7.2 — Predict: what happened to the function's name?

```python
def deco(func):
    def wrapper(*a, **k):
        return func(*a, **k)
    return wrapper

@deco
def greet():
    """Says hi."""
    return "hi"

print(greet.__name__)    # ?
print(greet.__doc__)     # ?
```

In [ ]:
def deco(func):
    def wrapper(*a, **k):
        return func(*a, **k)
    return wrapper

@deco
def greet():
    """Says hi."""
    return "hi"

print(greet.__name__)
print(greet.__doc__)

<details>
<summary>▶ Reveal</summary>

`wrapper`, and `None`.

Because `greet` is now `wrapper` (7.1), its identity *is* the wrapper's: `__name__` is `"wrapper"`, the
docstring is gone. This breaks introspection, debugging, and docs tooling — everything that reads a
function's metadata now sees the wrapper instead of `greet`. The fix is `functools.wraps`, which copies
the original's metadata onto the wrapper. The next exercise builds the corrected version.

</details>

## 7.3 — Build: a call-counting decorator that preserves metadata

Build `counted` so the decorated function (a) still works, (b) tracks how many times it's been called
on a `.calls` attribute of the wrapper, and (c) preserves the original's `__name__` via
`functools.wraps`. The counter is per-decorated-function state (a closure or a wrapper attribute).

In [ ]:
from functools import wraps

def counted(func):
    @wraps(func)
    def wrapper(*a, **k):
        # YOUR CODE HERE — increment wrapper.calls, then call func and return its result
        pass
    wrapper.calls = 0
    return wrapper

@counted
def add(x, y):
    """Adds two numbers."""
    return x + y

# Tests
assert add(2, 3) == 5
assert add(10, 20) == 30
assert add.calls == 2                 # state tracked on the wrapper
assert add.__name__ == "add"          # metadata preserved by wraps
assert add.__doc__ == "Adds two numbers."
print("7.3 passed")

<details>
<summary>▶ Note</summary>

`wrapper.calls` is state attached to the wrapper function object itself — functions are objects, so you
can hang attributes on them. (A closure variable would also work but couldn't be read from outside as
`add.calls`.) `@wraps(func)` copies `__name__`, `__doc__`, etc. from the original onto `wrapper`, so
`add` still looks like `add` to any tool inspecting it. Always use `wraps` in real decorators.

</details>

## 7.4 — Build: a decorator that takes an argument (three layers)

A decorator with its own argument needs **three** nested functions: outer takes the argument, middle
takes the function, inner is the wrapper. Build `repeat(n)` so the decorated function runs `n` times
and returns the list of results.

In [ ]:
from functools import wraps

def repeat(n):
    def decorator(func):
        @wraps(func)
        def wrapper(*a, **k):
            # YOUR CODE HERE — call func n times, collect results
            pass
        return wrapper
    return decorator

@repeat(3)
def roll():
    return "x"


# Tests
assert roll() == ["x", "x", "x"]

@repeat(2)
def echo(s):
    return s.upper()
assert echo("hi") == ["HI", "HI"]
print("7.4 passed")

<details>
<summary>▶ Why three layers</summary>

`@repeat(3)` is `repeat(3)` *called first*, returning a decorator, which then decorates the function.
So: `repeat(3)` → returns `decorator`; `decorator(roll)` → returns `wrapper`; `roll` becomes `wrapper`.
Three layers because there are three distinct inputs to thread: the decorator's argument (`n`), the
function (`func`), and the call's arguments (`*a, **k`). Each nested function captures one. This is the
shape of any configurable decorator — `@cache(maxsize=100)`, `@retry(times=3)`, etc.

</details>

## 7.5 — Predict: stacking order

```python
def tag(label):
    def deco(func):
        def wrapper(*a, **k):
            return f"<{label}>{func(*a, **k)}</{label}>"
        return wrapper
    return deco

@tag("outer")
@tag("inner")
def text():
    return "hi"

print(text())     # ? which tag ends up outermost
```

In [ ]:
def tag(label):
    def deco(func):
        def wrapper(*a, **k):
            return f"<{label}>{func(*a, **k)}</{label}>"
        return wrapper
    return deco

@tag("outer")
@tag("inner")
def text():
    return "hi"

print(text())

<details>
<summary>▶ Reveal</summary>

`<outer><inner>hi</inner></outer>`.

Decorators **apply bottom-up** but **execute top-down**. Application: `text` is first wrapped by the
nearest decorator (`tag("inner")`), then that result is wrapped by `tag("outer")` — so it's
`tag("outer")(tag("inner")(text))`. At call time the outermost wrapper runs first, wraps around the
inner one, which wraps around the original. Hence `outer` is on the outside of the output. The rule: the
decorator **closest to the function** wraps first (innermost); the one **on top** runs first at call
time (outermost). Getting this order right matters when you stack, say, `@cache` over `@validate` —
you want validation to run before the cache lookup, or after, depending on intent.

</details>

## ★ Milestone 07 — A cooperating decorator stack

Synthesize: build three real decorators and stack them on one function so they cooperate. Build:

- `count_calls(func)` — tracks `.calls` on the wrapper (from 7.3)
- `cache(func)` — memoizes by the single integer argument (repeat calls with the same arg skip the body)
- they should both use `functools.wraps`

Then stack `@count_calls` over `@cache` on a `square(n)` function that logs each *actual* computation,
and verify the interplay: the cache prevents recomputation, while `count_calls` counts *every* call
(cached or not). Reason about the stacking order from 7.5 to predict the counts.

(build it below)

Stack them like:
```python
@count_calls
@cache
def square(n): ...
```


In [ ]:
from functools import wraps

def count_calls(func):
    @wraps(func)
    def wrapper(*a, **k):
        # YOUR CODE HERE — increment wrapper.calls, call func
        pass
    wrapper.calls = 0
    return wrapper

def cache(func):
    store = {}
    @wraps(func)
    def wrapper(n):
        # YOUR CODE HERE — return cached if seen, else compute+store
        pass
    return wrapper

COMPUTED = []

@count_calls
@cache
def square(n):
    COMPUTED.append(n)
    return n * n

# Tests
COMPUTED.clear()
assert square(4) == 16
assert square(4) == 16      # cached — body NOT re-run
assert square(5) == 25
assert square(4) == 16      # cached again

# the body ran once per DISTINCT input
assert COMPUTED == [4, 5]

# count_calls is the OUTER decorator, so it counts EVERY call (cached or not)
assert square.calls == 4

# wraps preserved the name through both layers
assert square.__name__ == "square"
print("Milestone 07 passed")

<details>
<summary>▶ The interplay</summary>

Stacking is `count_calls(cache(square))` (7.5): `cache` wraps `square` first (innermost), then
`count_calls` wraps that (outermost). At call time the outer `count_calls` runs first and increments on
*every* call — so `.calls == 4` (all four invocations). It then delegates inward to `cache`, which only
runs the real `square` body for *new* arguments — so `COMPUTED == [4, 5]` (two distinct inputs). That's
why the counts differ: counting happens outside the cache, computation happens inside it. Swap the
order (`@cache` over `@count_calls`) and `.calls` would only count cache *misses* — a different, also
useful behavior. Choosing the stack order *is* the design decision. This is precisely how you'd layer
registration, validation, and caching on the swappable components later.

</details>